In [9]:
import os
import warnings
import gsw
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import xarray as xr
from crocolaketools import db_params

In [10]:
#loading spots
spots = pd.read_csv("/Users/kaleaholdren/Downloads/spots.csv")

#removing missing data
spots = spots.replace(-999, np.nan)
spots = spots.replace(-999.0, np.nan)

In [11]:
#creating a subset of spots
spots_subset = spots.iloc[::5500] #trying to get variety in time series site (iloc instead of just spots[1:20]
spots_subset

,TimeSeriesSite,CRUISE,STNNBR,CASTNO,BTLNBR,DATE,TIME,LATITUDE,LONGITUDE,CTDPRS,...,POP_FLAG_W,POP_SOPf,POP_P,POP_A,DOC,DOC_FLAG_W,DOC_SOPf,DOC_P,DOC_A,DOI
0,ALOHA,32MW001/1,1,3.0,12.0,19881031,56.0,22.7600,-157.9983,4.700,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1
5500,ALOHA,32WC031/1,31,6.0,18.0,19911021,1325.0,22.7500,-158.0000,111.700,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1
11000,ALOHA,32MW052/1,52,11.0,21.0,19940218,57.0,22.7903,-157.9825,18.800,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1
16500,ALOHA,32MW068/1,68,11.0,20.0,19951117,1955.0,22.7732,-158.0458,24.600,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1
22000,ALOHA,32MW090/1,90,12.0,4.0,19980220,157.0,22.6650,-158.0388,174.300,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1
27500,ALOHA,33KI113/1,113,9.0,2.0,20000329,1556.0,22.7580,-157.9705,799.500,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1
33000,ALOHA,33KI134/1,134,13.0,1.0,20020117,255.0,22.7798,-158.0275,1019.700,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1
38500,ALOHA,33KI158/1,158,10.0,7.0,20040421,1755.0,22.7343,-157.9745,73.800,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1
44000,ALOHA,33KB177/1,177,7.0,13.0,20060125,2336.0,22.7503,-157.9827,88.600,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1
49500,ALOHA,33KB192/1,192,3.0,18.0,20070609,1405.0,22.7502,-158.0003,1600.200,...,9,0,NaN,NaN,NaN,9,0,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1


In [9]:
#looking at date and time columns in my subset
spots_subset[["DATE", "TIME"]]

,DATE,TIME
0,19881031,56.0
5500,19911021,1325.0
11000,19940218,57.0
16500,19951117,1955.0
22000,19980220,157.0
27500,20000329,1556.0
33000,20020117,255.0
38500,20040421,1755.0
44000,20060125,2336.0
49500,20070609,1405.0


In [13]:
#trying first to create a function to standardize the date/time
def standardize_date(ddf): #add self as other input when doing it fr

    print("Converting SPOTS multiple time columns to one datetime")
    #converting date from float to string
    date_str = ddf["DATE"].astype("Int64").astype(str)
    #converting time from float to string & filling in zeros on left
    time_str = ddf["TIME"].astype("Int64").astype(str).str.zfill(4)

    ddf["JULD"] = pd.to_datetime(
        date_str + time_str,
        format = "%Y%m%d%H%M",
        errors = "coerce"
        #if I do dask then I need to add .persist() here?
    )

    
    #date when SPOTS csv file was generated
    ddf["date_update"] = np.datetime64("2024-02-22T00:00:00.00000000")
    
    return ddf

In [13]:
standardize_date(spots_subset)

Converting SPOTS multiple time columns to one datetime


/var/folders/nz/gjy99k0x7c98ysmfrlxzgl140000gn/T/ipykernel_14471/876127274.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ddf["JULD"] = pd.to_datetime(
/var/folders/nz/gjy99k0x7c98ysmfrlxzgl140000gn/T/ipykernel_14471/876127274.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ddf["date_update"] = np.datetime64("2024-02-22T00:00:00.00000000")


,TimeSeriesSite,CRUISE,STNNBR,CASTNO,BTLNBR,...,DOC_P,DOC_A,DOI,JULD,date_update
0,ALOHA,32MW001/1,1,3.0,12.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,1988-10-31 00:56:00,2024-02-22
5500,ALOHA,32WC031/1,31,6.0,18.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,1991-10-21 13:25:00,2024-02-22
11000,ALOHA,32MW052/1,52,11.0,21.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,1994-02-18 00:57:00,2024-02-22
16500,ALOHA,32MW068/1,68,11.0,20.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,1995-11-17 19:55:00,2024-02-22
22000,ALOHA,32MW090/1,90,12.0,4.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,1998-02-20 01:57:00,2024-02-22
27500,ALOHA,33KI113/1,113,9.0,2.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,2000-03-29 15:56:00,2024-02-22
33000,ALOHA,33KI134/1,134,13.0,1.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,2002-01-17 02:55:00,2024-02-22
38500,ALOHA,33KI158/1,158,10.0,7.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,2004-04-21 17:55:00,2024-02-22
44000,ALOHA,33KB177/1,177,7.0,13.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,2006-01-25 23:36:00,2024-02-22
49500,ALOHA,33KB192/1,192,3.0,18.0,...,NaN,NaN,https://doi.org/10.1575/1912/bco-dmo.3773.1,2007-06-09 14:05:00,2024-02-22


In [14]:
#keep best values function
#we know FLAG_W = 2 means usable
#SOPf = 1 means meets required & desired, =2 means meets required
def keep_best_values(df, param):
    #df = a row or partition of pandas dataframe
    #param = name of the QC variable of the parameter
    #returns updated dataframe

    #2 means usable (assuming we are only using FLAG_W, not SOPf)
    condition = ~df[param].isin([2])

    #find bad QC values
    df.loc[condition, param] = pd.NA
    #remove "_FLAG_W"
    df.loc[condition, param[:-7]] = pd.NA

    return df

In [15]:
#testing keep_best_values
#but how do I test name part?
test = spots_subset.copy()
keep_best_values(test, "NITRAT_FLAG_W")
test["NITRAT_FLAG_W"]

0         2.0
5500      NaN
11000     NaN
16500     NaN
22000     NaN
27500     NaN
33000     NaN
38500     NaN
44000     NaN
49500     2.0
55000     NaN
60500     NaN
66000     NaN
71500     2.0
77000     NaN
82500     NaN
88000     2.0
93500     2.0
99000     2.0
104500    2.0
Name: NITRAT_FLAG_W, dtype: float64

In [6]:
#this section of "def standardize_data(self,ddf)" 
#this has not been tested, don't think I can at this point
def keep_good_values(ddf):
    #keep only good QC values
    params_to_check =[]
    for param in db_params.params["SPOTS2CROCOLAKE"].keys(): 
        if param.endswith("_FLAG_W") and param in ddf.columns:
            ddf = ddf.apply(
                self.keep_best_values, param
            )
            params_to_check.append(param[:-7])
            
    #remove rows containing all NAs
    ddf = ddf.apply( #apply instead of map_partitions bc not dask
        super().remove_all_NAs, params_to_check
    )
    
    return ddf

In [3]:
# #making dictionary
# #needs to go in db_params.py

# #original names of parameters to keep
# params["SPOTS"] = [
#     'DATE',
#     'TIME',
#     'LATITUDE',
#     'LONGITUDE',
#     'CTDPRS'
#     'CTDTEMP',
#     #'CTDSAL',
#     #'CTDOXY',
#     #'SALNTY',
#     #'OXYGEN',
#     'NITRAT',
#     'PHSPHT',
#     'SILCAT',
#     'ALKALI',
#     'PH_TOT',
# ]

# #dict for renaming parameters to crocolake names
# params["SPOTS2CROCOLAKE"] = {
#     #??expocode : 'PLATFORM_NUMBER': ???
#     'LATITUDE' : 'LATITUDE',
#     'LONGITUDE' : 'LONGITUDE',
#     'CTDPRES' : 'PRES',
#     'CTDTMP' : 'TEMP',
#     #'CTDSAL' or 'SALNTY' : 'PSAL',
#     #'CTDOXY' or 'OXYGEN' : 'DOXY',
#     'NITRAT' : 'NITRATE',
#     'SILCAT' : 'SILICATE',
#     'PHSPHT' : 'PHOSPHATE',
#     'ALKALI' : 'TOT_ALKALINITY',
#     'PH_TOT' : 'PH_IN_SITU_TOTAL',
#     ##'CTDSAL_FLAG_W' or 'SALNTY_FLAG_W' : 'PSAL_QC',
#     #'CTDOXY_FLAG_W' or 'OXYGEN_FLAG_W' : 'DOXY_QC',
#     'NITRAT_FLAG_W' : 'NITRATE_QC',
#     'SILCAT_FLAG_W' : 'SILICATE_QC',
#     'PHSPHT_FLAG_W' : 'PHOSPHATE_QC',
#     'ALKALI_FLAG_W' : 'TOT_ALKALINITY_QC',
#     'PH_TOT_FLAG_W' : 'PH_IN_SITU_TOTAL_QC',

#     'profile_nb' : 'CYCLE_NUMBER', #temporary name for profile ID, this is created in converter?
#     'date_update' : 'DATE_UPDATE' #temporary name for date update
# }

In [7]:
def read_to_df(self, filename = None, lock = None):
    """Read file into a pandas dataframe

    Argument:
    filename -- file name, including relative path

    Returns
    df -- pandas dataframe
    """

    if filename is None:
        filename = "SPOTS???.csv" #what should the name be?
        print("Using default filename: ", filename)

    input_fname = self.input_path + filename
    print("Reading SPOTS file: ", input_fname)

    #low_memory = False as SPOTS is a small db (I think --check this)
    ddf = pd.read_csv( #change to dd.read_csv later on?
        input_fname,
        assume_missing = True,
        delimiter = ",",
        header = 0,
        low_memory = False,
        dtype_backend = 'pyarrow'
    )

    return self.standardize_data(ddf) #don't have full standardize_data() yet

In [14]:
#testing that JULD worked
spots_subset = spots_subset.copy()
spots_subset = standardize_date(spots_subset)

spots_subset[["DATE", "TIME", "JULD", "date_update"]].head(20)

Converting SPOTS multiple time columns to one datetime


,DATE,TIME,JULD,date_update
0,19881031,56.0,1988-10-31 00:56:00,2024-02-22
5500,19911021,1325.0,1991-10-21 13:25:00,2024-02-22
11000,19940218,57.0,1994-02-18 00:57:00,2024-02-22
16500,19951117,1955.0,1995-11-17 19:55:00,2024-02-22
22000,19980220,157.0,1998-02-20 01:57:00,2024-02-22
27500,20000329,1556.0,2000-03-29 15:56:00,2024-02-22
33000,20020117,255.0,2002-01-17 02:55:00,2024-02-22
38500,20040421,1755.0,2004-04-21 17:55:00,2024-02-22
44000,20060125,2336.0,2006-01-25 23:36:00,2024-02-22
49500,20070609,1405.0,2007-06-09 14:05:00,2024-02-22


In [16]:
#looking at failed rows for date & time: do I need to put midnight for these?
spots_subset.loc[
    spots_subset["JULD"].isna(),
    ["DATE", "TIME"]
]

,DATE,TIME
93500,20061017,NaN
104500,20161119,NaN
